In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

os.chdir("/home/dermodkkelly/rumen_microbiome_pipeline")

OUTDIR = Path("postprocessing/output")

genus = pd.read_csv(
    OUTDIR / "genus_counts_no_controls_515F_806R.csv"
)

genus.head()


,SampleID,0319-6G20,0319-7L14,11-24,1174-901-12,67-14,A4b,ADurb.Bin063-1,ASF356,Abditibacterium,...,mle1-7,p-1088-a5_gut_group,p-251-o5,p-2534-18B5_gut_group,possible_genus_Sk018,probable_genus_10,uncultured,vadinBA26,vadinBE97,vadinHA49
0,Tully__10732_S4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,238.0,45.0,0.0,0.0,36.0,1119.0,0.0,91.0,0.0
1,Tully__10777_S5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,144.0,160.0,0.0,0.0,349.0,1265.0,0.0,75.0,0.0
2,Tully__10785_S6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,119.0,191.0,0.0,0.0,160.0,3133.0,0.0,206.0,0.0
3,Tully__11203_S10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,29.0,0.0,2.0,0.0,0.0,627.0,0.0,11.0,0.0
4,Tully__11308_S12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,179.0,24.0,3.0,0.0,273.0,2140.0,0.0,29.0,0.0


In [2]:
# --- metadata vs feature columns ---
# combined data has ONLY SampleID as metadata (animal info joined later)
meta_cols = ["SampleID"]
feature_cols = [c for c in genus.columns if c not in meta_cols]
print(len(feature_cols))

858


In [3]:
# --- metadata vs feature columns ---
meta_cols = ["SampleID"]
feature_cols = [c for c in genus.columns if c not in meta_cols]

print("Starting genus-level features:", len(feature_cols))


# --- derive host group from SampleID ---
sample_info = genus[["SampleID"]].copy()

sample_info["batch"] = (
    sample_info["SampleID"]
    .str.extract(r"^([^_]+)", expand=False)
)

sample_info["stem"] = (
    sample_info["SampleID"]
    .str.replace(r"^[^_]+__", "", regex=True)
)

sample_info["cohort"] = (
    sample_info["stem"]
    .str.replace(r"_?[0-9]+$", "", regex=True)
)


def assign_species(row):
    cohort = row["cohort"]
    batch = row["batch"]

    if str(cohort).startswith("Sheep"):
        return "Sheep"
    elif str(cohort).startswith("Beef"):
        return "Beef"
    elif str(cohort).startswith("Dairy"):
        return "Dairy"
    elif batch in ["EN00010710", "EN00012132"]:
        return "Dairy"
    elif batch == "CTmicro":
        return "Sheep"
    else:
        return "Other"


sample_info["species"] = sample_info.apply(assign_species, axis=1)

print(sample_info["species"].value_counts())

Starting genus-level features: 858
species
Sheep    1126
Other     278
Dairy     165
Beef      164
Name: count, dtype: int64


In [4]:
# --- prevalence calculated independently within each target host group ---

target_hosts = ["Sheep", "Beef", "Dairy"]

prev_by_host = {}

for host in target_hosts:
    idx = sample_info["species"] == host

    prev_by_host[host] = (
        genus.loc[idx, feature_cols] > 0
    ).mean()

prev_by_host = pd.DataFrame(prev_by_host)


# retain a taxon if prevalence >=10% in AT LEAST ONE host group
keep = prev_by_host.index[
    (prev_by_host >= 0.10).any(axis=1)
].tolist()

print("Features retained at >=10% prevalence in at least one host group:",
      len(keep))


# useful audit
print("\nFeatures >=10% prevalence within each host group:")
for host in target_hosts:
    print(
        host,
        (prev_by_host[host] >= 0.10).sum()
    )


# how many pass in 1, 2, or all 3 groups?
n_hosts_pass = (prev_by_host >= 0.10).sum(axis=1)

print("\nNumber of host groups in which each retained feature passes 10%:")
print(n_hosts_pass[n_hosts_pass > 0].value_counts().sort_index())

# Overall prevalence across all samples (used in the feature summary below;
# prev_by_host above is per host group, this is the pooled figure)
prev = (genus[feature_cols] > 0).mean()


Features retained at >=10% prevalence in at least one host group: 238

Features >=10% prevalence within each host group:
Sheep 191
Beef 196
Dairy 199

Number of host groups in which each retained feature passes 10%:
1     50
2     28
3    160
Name: count, dtype: int64


In [5]:
X = genus[keep].astype(float) + 1e-6

logX = np.log(X)

clr = logX.sub(logX.mean(axis=1), axis=0)

np.allclose(clr.sum(axis=1), 0)

True

In [6]:
# CLR table
clr_df = pd.concat([genus[meta_cols], clr], axis=1)

clr_df.to_csv(
    OUTDIR / "CLR_genus_only_515F_806R.csv",
    index=False
)

# raw prevalence-filtered counts
raw_filtered = pd.concat(
    [genus[meta_cols], genus[keep]],
    axis=1
)

raw_filtered.to_csv(
    OUTDIR / "raw_filtered_genus_515F_806R.csv",
    index=False
)

# feature summary
summary = pd.DataFrame({
    "Taxon": keep,
    "Prevalence": prev[keep].values,
    "TotalCount": genus[keep].sum().values
})

summary.sort_values(
    "TotalCount",
    ascending=False
).to_csv(
    OUTDIR / "feature_summary_515F_806R.csv",
    index=False
)